# Statistical Learning 2 — Possíveis Questões para o Exame
**FH Kufstein Tirol | SS 2026**

---

> **Como usar este arquivo:**  
> Cada exercício apresenta as questões no **estilo da Probeklausur** (mesma estrutura, pontuação e formato).  
> Tente responder cada questão **sem olhar a resposta** logo abaixo.  
> As respostas indicam os **pontos-chave que o examinador espera** — não apenas o resultado, mas a justificativa e o raciocínio por trás de cada equação.

---

## Cobertura por Aula

| Exercício | Aula | Tópico | Pontos |
|-----------|------|--------|--------|
| 1 | L1 | Modelos Lineares Gerais e Funções de Base | 20 |
| 2 | L2 | Decomposição Bias-Variância e Seleção de Modelos | 20 |
| 3 | L3 | Regularização: Tikhonov, L1/L2, Early Stopping, CV | 20 |
| 4 | L4 | Visão Probabilística e Regularização Bayesiana | 20 |
| 5 | L1–L4 | Questões Aplicadas e Conceituais | 10 |
| Bônus | L3+L4 | Bootstrap e Cross-Validation Avançado | 10 |
| **Total** | | | **90 + 10 bônus = 100** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import KFold, cross_val_score
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})
print('Bibliotecas carregadas com sucesso.')

---
---
# EXERCÍCIO 1 — (20 pontos)
**Tema: Modelos Lineares Gerais e Funções de Base**

Seja $\{(x_1, y_1), \ldots, (x_n, y_n)\} \subset \mathbb{R}^2$ um conjunto de dados e $B_1, \ldots, B_p : \mathbb{R} \to \mathbb{R}$ funções de base fixadas.

**(a)** Escreva o modelo ansatz para a **regressão linear geral**. Defina a matriz de design $\mathbf{B} \in \mathbb{R}^{n \times p}$ e o vetor de parâmetros $\boldsymbol{\theta}$. **(2)**

**(b)** Explique o que significa dizer que o modelo é *linear nos parâmetros* mas possivelmente não-linear em $x$. Por que essa propriedade é importante? **(2)**

**(c)** Derive a **Equação Normal** a partir do princípio dos mínimos quadrados. Mostre todos os passos: definição da perda, cálculo do gradiente e solução. **(4)**

**(d)** Descreva **três tipos diferentes de funções de base** (polinomial, RBF Gaussiana, Fourier). Para cada uma, escreva a expressão matemática e indique em que tipo de problema é preferível. **(4)**

**(e)** Quando a matriz $\mathbf{B}^\top\mathbf{B}$ é **singular** (não invertível)? Cite duas situações práticas que causam isso, e explique como a regularização Ridge resolve o problema. **(4)**

**(f)** Mostre como a regressão polinomial de grau $d$ em uma variável $x$ é um **caso especial** do modelo linear geral. Escreva explicitamente a matriz de design $\mathbf{B}$ para $n=3$ amostras e $d=2$. **(4)**

## ✅ Resposta — Exercício 1

---

### (a) Modelo Ansatz da Regressão Linear Geral

$$\boxed{f(x) = \sum_{j=1}^{p} \theta_j B_j(x) = \mathbf{B}(x)^\top \boldsymbol{\theta}}$$

- $\boldsymbol{\theta} = (\theta_1, \ldots, \theta_p)^\top \in \mathbb{R}^p$ — **vetor de parâmetros** (o que queremos aprender)
- $\mathbf{B}(x) = (B_1(x), \ldots, B_p(x))^\top$ — **funções de base** avaliadas em $x$ (fixadas a priori)

A **matriz de design** $\mathbf{B} \in \mathbb{R}^{n \times p}$ tem entradas $\mathbf{B}_{ij} = B_j(x_i)$ — a $i$-ésima amostra avaliada na $j$-ésima base:

$$\mathbf{B} = \begin{pmatrix} B_1(x_1) & B_2(x_1) & \cdots & B_p(x_1) \\ B_1(x_2) & B_2(x_2) & \cdots & B_p(x_2) \\ \vdots & \vdots & \ddots & \vdots \\ B_1(x_n) & B_2(x_n) & \cdots & B_p(x_n) \end{pmatrix}$$

O modelo para todo o dataset compactamente: $\hat{\mathbf{y}} = \mathbf{B}\boldsymbol{\theta}$.

> **Intuição:** Pense nas funções de base como "blocos de construção" de formas diferentes (curvas simples, ondas, gaussianas). O modelo combina esses blocos com pesos $\theta_j$. Aprender $f$ significa descobrir quais blocos são mais importantes e em qual proporção.

---

### (b) Linear nos parâmetros vs. não-linear em $x$

**Linear nos parâmetros** significa que $f(x) = \sum_j \theta_j B_j(x)$ é uma combinação linear de $\boldsymbol{\theta}$ — cada $\theta_j$ aparece multiplicado por um fator que não depende de outros $\theta$'s.

Exemplo: $f(x) = \theta_1 x + \theta_2 x^2 + \theta_3 x^3$
- **Linear em $\boldsymbol{\theta}$:** sim — $f(x)$ é uma soma de $\theta_j \cdot(\text{algo})$
- **Linear em $x$:** não — a função de $x$ é cúbica

**Por que isso importa?** A linearidade em $\boldsymbol{\theta}$ torna o problema de otimização (minimizar o MSE) uma função **quadrática convexa** de $\boldsymbol{\theta}$, com solução analítica fechada: a Equação Normal. Sem linearidade (como em redes neurais), a otimização é não-convexa e requer métodos iterativos sem garantia de ótimo global.

---

### (c) Equação Normal — Derivação completa

**Objetivo:** Encontrar $\boldsymbol{\theta}^*$ que minimize o erro quadrático total.

**Passo 1 — Função de perda (Mínimos Quadrados):**
$$\mathcal{L}(\boldsymbol{\theta}) = \|\mathbf{y} - \mathbf{B}\boldsymbol{\theta}\|^2 = (\mathbf{y} - \mathbf{B}\boldsymbol{\theta})^\top(\mathbf{y} - \mathbf{B}\boldsymbol{\theta})$$

> Por que o quadrado? É diferenciável em todo ponto (ao contrário de $|y - \hat{y}|$), convexo (garante mínimo global único se $\mathbf{B}^\top\mathbf{B}$ é invertível) e penaliza erros grandes mais severamente.

**Passo 2 — Expandir:**
$$\mathcal{L}(\boldsymbol{\theta}) = \mathbf{y}^\top\mathbf{y} - 2\boldsymbol{\theta}^\top\mathbf{B}^\top\mathbf{y} + \boldsymbol{\theta}^\top\mathbf{B}^\top\mathbf{B}\boldsymbol{\theta}$$

**Passo 3 — Gradiente e condição de ótimo ($\nabla_\theta \mathcal{L} = 0$):**
$$\nabla_{\boldsymbol{\theta}} \mathcal{L} = -2\mathbf{B}^\top\mathbf{y} + 2\mathbf{B}^\top\mathbf{B}\boldsymbol{\theta} = \mathbf{0}$$

> Em um ponto de mínimo, a função não cresce nem decresce em nenhuma direção — o gradiente é zero. Análogo ao ponto mais baixo de uma bacia parabólica.

**Passo 4 — Resolver:**
$$\mathbf{B}^\top\mathbf{B}\,\boldsymbol{\theta} = \mathbf{B}^\top\mathbf{y} \quad \Longrightarrow \quad \boxed{\boldsymbol{\theta}^* = (\mathbf{B}^\top\mathbf{B})^{-1}\mathbf{B}^\top\mathbf{y}}$$

---

### (d) Três tipos de funções de base

| Base | Expressão $B_j(x)$ | Quando usar |
|------|-------------------|-------------|
| **Polinomial** | $x^{j-1}$ (para $j=1,\ldots,p$) | Dados suaves e sem periodicidade; simples e interpretável; cuidado com instabilidade numérica para $p$ grande |
| **Gaussiana (RBF)** | $\exp\!\left(-\frac{\|x - c_j\|^2}{2h^2}\right)$ | Dados com estrutura local, picos, clusters; $c_j$ são centros, $h$ controla a largura; muito usada em kernel methods |
| **Fourier** | $\sin(j\omega x)$ e $\cos(j\omega x)$ | Dados **periódicos** (sinais, séries temporais cíclicas, dados sazonais); cada par de base captura uma frequência $j\omega$ |

Intuição: a base polinomial é global (muda o modelo em todo o domínio), a RBF é local (afeta apenas a vizinhança de $c_j$), a Fourier captura periodicidade.

---

### (e) Quando $\mathbf{B}^\top\mathbf{B}$ é singular?

$\mathbf{B}^\top\mathbf{B}$ é singular (não invertível) quando suas colunas são linearmente dependentes — ou seja, quando existe $\boldsymbol{v} \neq \mathbf{0}$ tal que $\mathbf{B}^\top\mathbf{B}\,\boldsymbol{v} = 0$.

**Situações práticas que causam isso:**

1. **$p > n$ (mais parâmetros do que amostras):** A matriz $\mathbf{B} \in \mathbb{R}^{n \times p}$ com $p > n$ tem no máximo $n$ colunas linearmente independentes → $\text{rank}(\mathbf{B}) \leq n < p$ → $\mathbf{B}^\top\mathbf{B}$ singular. O sistema tem infinitas soluções.

2. **Multicolinearidade (features quase iguais):** Se dois grupos de features são quase linearmente dependentes (ex: $B_2(x) \approx 2 \cdot B_1(x)$), as colunas correspondentes de $\mathbf{B}$ são quase paralelas → $\mathbf{B}^\top\mathbf{B}$ é mal condicionada (quase singular) → solução numérica instável.

**Como a Ridge resolve:**
$$(\mathbf{B}^\top\mathbf{B} + \lambda I)^{-1}\mathbf{B}^\top\mathbf{y}$$
Somar $\lambda I$ desloca todos os autovalores de $\mathbf{B}^\top\mathbf{B}$ por $+\lambda$. Como $\lambda > 0$, todos os autovalores ficam $> 0$ → a matriz torna-se **definida positiva** → sempre invertível, independente de $p$, $n$ ou multicolinearidade.

---

### (f) Regressão Polinomial como caso especial

A regressão polinomial de grau $d$ usa as funções de base $B_j(x) = x^{j-1}$ para $j = 1, \ldots, d+1$ (com $p = d+1$ bases, incluindo o intercepto $B_1(x) = x^0 = 1$).

O modelo é: $f(x) = \theta_1 + \theta_2 x + \theta_3 x^2 + \cdots + \theta_{d+1} x^d$

Para $n=3$ amostras $\{x_1, x_2, x_3\}$ e grau $d=2$ (logo $p=3$):

$$\mathbf{B} = \begin{pmatrix} 1 & x_1 & x_1^2 \\ 1 & x_2 & x_2^2 \\ 1 & x_3 & x_3^2 \end{pmatrix} \in \mathbb{R}^{3 \times 3}$$

Esta é a clássica **matriz de Vandermonde** — é invertível se e somente se os $x_i$ são distintos, garantindo solução única para $p = n = 3$.

In [ ]:
# === Visualização — Exercício 1: Funções de Base e Ajuste pelo Equação Normal ===

np.random.seed(0)
x_data = np.sort(np.random.uniform(0, 2*np.pi, 25))
y_data = np.sin(x_data) + np.random.normal(0, 0.2, 25)
x_plot = np.linspace(0, 2*np.pi, 300)

def design_matrix(x, p):
    return np.column_stack([x**j for j in range(p)])

def normal_eq(x, y, p, lam=0):
    B = design_matrix(x, p)
    return np.linalg.solve(B.T @ B + lam*np.eye(p), B.T @ y)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Funções de base
ax = axes[0]
cores = ['#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6']
for j, c in enumerate(cores):
    label = f'$B_1=1$' if j==0 else f'$B_{j+1}=x^{j}$'
    normed = x_plot**j / max(1, (2*np.pi)**j)
    ax.plot(x_plot, normed, color=c, lw=2, label=label)
ax.set_title('Funções de Base Polinomiais\n$B_j(x) = x^{j-1}$', fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('$B_j(x)$')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Matriz de design
ax = axes[1]
B_vis = design_matrix(x_data[:15], 5)
B_n = (B_vis - B_vis.min(0)) / (B_vis.max(0)-B_vis.min(0)+1e-8)
im = ax.imshow(B_n, aspect='auto', cmap='Blues')
ax.set_title('Matriz de Design $\\mathbf{B}\\in\\mathbb{R}^{n\\times p}$\n'
             'Entrada: $\\mathbf{B}_{ij}=B_j(x_i)$', fontweight='bold')
ax.set_xlabel('j (função de base)'); ax.set_ylabel('i (amostra)')
ax.set_xticks(range(5)); ax.set_xticklabels([f'$B_{j+1}$' for j in range(5)])
plt.colorbar(im, ax=ax)

# Ajuste: Equação Normal com diferentes p
ax = axes[2]
ax.scatter(x_data, y_data, color='black', s=35, zorder=5, label='Dados')
ax.plot(x_plot, np.sin(x_plot), 'k--', lw=1.5, alpha=0.5, label='$f^*=\\sin(x)$')
for p, cor, nome in [(2,'#E74C3C','p=2 (underfitting)'),(6,'#3498DB','p=6 (bom)'),(14,'#2ECC71','p=14 (overfitting)')]:
    theta = normal_eq(x_data, y_data, p, lam=1e-8)
    pred = np.clip(design_matrix(x_plot, p) @ theta, -2.5, 2.5)
    ax.plot(x_plot, pred, color=cor, lw=2, label=nome)
ax.set_title('Equação Normal: $\\hat{\\theta}=(\\mathbf{B}^\\top\\mathbf{B})^{-1}\\mathbf{B}^\\top\\mathbf{y}$',
             fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(-2, 2)

plt.suptitle('Exercício 1 — Regressão Linear Geral', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
---
# EXERCÍCIO 2 — (20 pontos)
**Tema: Decomposição Bias-Variância e Seleção de Modelos**

**(a)** Escreva a **decomposição Bias-Variância** do MSE esperado. Defina e explique intuitivamente cada termo. Relacione cada termo com o número de funções de base $p$. **(4)**

**(b)** Prove matematicamente que **a variância da média de $B$ estimadores independentes** é $B$ vezes menor que a variância de um único estimador. Que algoritmo de Machine Learning usa diretamente este princípio? **(4)**

**(c)** Descreva pelo menos **4 métodos** para reduzir a variância em um problema de aprendizado estatístico. Para cada um, explique o mecanismo e o efeito sobre o bias. **(4)**

**(d)** Explique como as **Learning Curves** (curvas de aprendizado em função de $N$) podem ser usadas para diagnosticar underfitting vs. overfitting. Descreva o padrão visual esperado para cada caso e o que fazer em cada situação. **(4)**

**(e)** O que é a **regra 1-SE** (one-standard-error rule) na seleção de modelo? Por que pode ser preferível ao mínimo absoluto do erro de CV? **(2)**

**(f)** Explique o que é o **erro de generalização** e por que o erro de treino o **subestima sistematicamente**. O que torna o erro de treino um estimador enviesado? **(2)**

## ✅ Resposta — Exercício 2

---

### (a) Decomposição Bias-Variância

$$\mathbb{E}\left[(y - \hat{f}(x))^2\right] = \underbrace{\left(f^*(x) - \mathbb{E}[\hat{f}(x)]\right)^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}\left[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2\right]}_{\text{Variância}} + \underbrace{\sigma_\varepsilon^2}_{\text{Ruído Irredutível}}$$

**Bias² (Viés):** Desvio sistemático — quão longe a predição *média* do modelo está da verdade real $f^*(x)$.
- Analogia: atirar flechas num alvo — o bias é o desvio sistemático do centro, independente de quantas flechas você atire.
- **p pequeno** (modelo simples) → alto bias → underfitting: não consegue capturar a estrutura dos dados.

**Variância:** Instabilidade do modelo — quanto a predição muda ao trocar o dataset de treino.
- Continuando a analogia: a variância é a dispersão das flechas — cada dataset produz um modelo diferente.
- **p grande** (modelo complexo) → alta variância → overfitting: memoriza o ruído do treino específico.

**Ruído irredutível $\sigma_\varepsilon^2$:** Inerente nos dados. Não pode ser reduzido por nenhum modelo — é o limite fundamental de predição (o piso do MSE).

| p pequeno | p grande |
|-----------|----------|
| Alto Bias, Baixa Variância | Baixo Bias, Alta Variância |
| Underfitting | Overfitting |

O $p^*$ ótimo minimiza Bias² + Variância e é encontrado por Cross-Validation.

---

### (b) Variância da média de $B$ estimadores

**Prova:** Sejam $\hat{f}^{(1)}, \ldots, \hat{f}^{(B)}$ estimadores independentes com a mesma variância $\sigma^2_f$.

$$\hat{f}_\text{bag} = \frac{1}{B}\sum_{b=1}^B \hat{f}^{(b)}$$

$$\text{Var}(\hat{f}_\text{bag}) = \text{Var}\!\left(\frac{1}{B}\sum_{b=1}^B \hat{f}^{(b)}\right) = \frac{1}{B^2}\sum_{b=1}^B \text{Var}(\hat{f}^{(b)}) = \frac{1}{B^2} \cdot B \cdot \sigma^2_f = \frac{\sigma^2_f}{B}$$

A segunda igualdade usa a independência entre os estimadores (covariância zero).

**Conclusão:** $\text{Var}(\hat{f}_\text{bag}) = \sigma^2_f / B$ — a variância cai por um fator de $B$.

O algoritmo que usa diretamente este princípio é o **Bagging (Bootstrap Aggregating)** — e sua extensão, o **Random Forest**. Treinar $B$ árvores de decisão (com alta variância individual) em amostras bootstrap e calcular a média produz um ensemble $B$ vezes mais estável.

---

### (c) Quatro métodos para reduzir a variância

**1. Regularização L2 (Ridge):** $\mathcal{L}_\text{reg} = \|\mathbf{B}\theta - y\|^2 + \lambda\|\theta\|^2$
- Encolhe todos os $\theta_j$ em direção a zero uniformemente — coeficientes menores → predições menos extremas → menor variância
- Efeito no bias: aumenta levemente (encolhe sinal + ruído)
- Solução analítica: $(\mathbf{B}^\top\mathbf{B} + \lambda I)^{-1}\mathbf{B}^\top y$

**2. Regularização L1 (Lasso):** $\mathcal{L}_\text{reg} = \|\mathbf{B}\theta - y\|^2 + \lambda\|\theta\|_1$
- Força muitos $\theta_j = 0$ exatamente → modelo mais simples → menor variância
- Bônus: feature selection automática (modelo esparso)
- Efeito no bias: pode aumentar ao zerar features relevantes

**3. Early Stopping:** Para o treinamento iterativo quando o erro de validação começa a crescer
- Mecanismo: iterações iniciais capturam estrutura real; iterações tardias memorizam ruído
- Equivalente matematicamente a Ridge com $\lambda \propto 1/t$
- Efeito no bias: iterações insuficientes → modelo subajustado → maior bias

**4. Bagging:** Treina $B$ modelos em amostras bootstrap e agrega
- $\hat{f}_\text{bag}(x) = \frac{1}{B}\sum_{b=1}^B \hat{f}^{(b)}(x)$
- Reduz variância por fator $B$ sem aumentar o bias (bias da média ≈ bias individual)
- Funciona melhor para modelos instáveis (alta variância individual, como árvores profundas)

---

### (d) Diagnóstico via Learning Curves

Uma **Learning Curve** plota o erro de treino e de validação em função de $N$ (tamanho do dataset de treino).

**Padrão: Alta Variância (overfitting)**
- Treino: baixo e quase plano conforme N cresce
- Validação: alto, mas decresce com N
- Gap grande entre as curvas mesmo para N grande
- O que fazer: mais dados ajuda gradualmente; aumentar $\lambda$; bagging; reduzir $p$

**Padrão: Alto Bias (underfitting)**
- Treino: alto e rapidamente "plana" com N
- Validação: alto e converge rapidamente para o erro de treino (gap pequeno)
- Mais dados **não ajuda** — ambas as curvas estabilizam em erro alto
- O que fazer: diminuir $\lambda$; mais features; bases mais expressivas; modelo mais complexo

> Resumo diagnóstico: **gap grande = variância; ambos altos = bias.**

---

### (e) Regra 1-SE (one-standard-error rule)

A **regra 1-SE** seleciona o modelo mais simples cujo erro de CV está dentro de **1 desvio padrão** do erro mínimo de CV:
$$\lambda^* = \max\{\lambda : \hat{\text{Err}}(\lambda) \leq \hat{\text{Err}}_\min + \hat{\text{se}}_\min\}$$

**Por que pode ser melhor que simplesmente minimizar?**
O erro de CV tem variabilidade estatística — a diferença entre o $\lambda$ com erro mínimo e um $\lambda$ ligeiramente maior pode ser estatisticamente insignificante. A regra 1-SE escolhe o modelo mais parcimonioso que é estatisticamente indistinguível do melhor, evitando selecionar $\lambda$ muito pequeno por acaso. Na prática, modelos mais simples generalizam melhor fora do dataset de validação.

---

### (f) Erro de generalização e viés do erro de treino

O **erro de generalização** é o MSE esperado em dados nunca vistos, provenientes da mesma distribuição que gerou o treino:
$$\text{Err} = \mathbb{E}_{(x,y) \sim P}\left[(y - \hat{f}(x))^2\right]$$

**Por que o erro de treino subestima sistematicamente?**

O modelo $\hat{f}$ foi **otimizado** para minimizar o erro *naquelas mesmas amostras* de treino. Ele "conhece" os dados de treino — memoriza padrões e até ruído. Então, avaliá-lo nas mesmas amostras é como corrigir uma prova com o gabarito na mão: o resultado é artificialmente bom.

Formalmente: $\mathbb{E}[\text{Err}_\text{treino}] < \mathbb{E}[\text{Err}_\text{generalização}]$.

A diferença é chamada de **pessimismo** ou **otimismo** do estimador de treino:
$$\text{otimismo} = \mathbb{E}[\text{Err}_\text{gen}] - \mathbb{E}[\text{Err}_\text{treino}] = \frac{2}{n}\sum_{i=1}^n \text{Cov}(y_i, \hat{f}(x_i))$$

Quanto mais flexível o modelo, maior a covariância entre $y_i$ e $\hat{f}(x_i)$ → maior o otimismo. Cross-Validation corrige esse viés ao avaliar em dados que o modelo não viu.

In [ ]:
# === Visualização — Exercício 2: Bias-Variância e Learning Curves ===

def f_star(x): return np.sin(2*x) + 0.4*x

x_all = np.linspace(0, 3, 250)
n_trials, n_pts, sigma = 60, 30, 0.4

def fit_poly(x_tr, y_tr, p, lam=1e-7):
    B = np.column_stack([(x_tr/3)**j for j in range(p+1)])
    return np.linalg.solve(B.T @ B + lam*np.eye(p+1), B.T @ y_tr)

degrees = [1, 5, 11]
preds = {d: np.zeros((n_trials, len(x_all))) for d in degrees}

for t in range(n_trials):
    x_t = np.sort(np.random.uniform(0, 3, n_pts))
    y_t = f_star(x_t) + np.random.normal(0, sigma, n_pts)
    for d in degrees:
        B_te = np.column_stack([(x_all/3)**j for j in range(d+1)])
        th = fit_poly(x_t, y_t, d)
        preds[d][t] = np.clip(B_te @ th, -4, 6)

labels = {1:'p=2 — Alto Bias, Baixa Variância',5:'p=6 — Equilíbrio',11:'p=12 — Baixo Bias, Alta Variância'}
colors = {1:'#E74C3C',5:'#2ECC71',11:'#3498DB'}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for col, d in enumerate(degrees):
    ax = axes[0, col]
    mu = preds[d].mean(0); sd = preds[d].std(0)
    for t in range(15):
        ax.plot(x_all, preds[d][t], alpha=0.12, color=colors[d], lw=1)
    ax.plot(x_all, mu, color=colors[d], lw=2.5, label='Predição média')
    ax.fill_between(x_all, mu-sd, mu+sd, alpha=0.25, color=colors[d], label='± 1 std')
    ax.plot(x_all, f_star(x_all), 'k--', lw=2, label='$f^*(x)$')
    ax.set_title(labels[d], fontweight='bold', fontsize=9)
    ax.set_ylim(-3, 4); ax.legend(fontsize=7); ax.grid(alpha=0.3)

# Curva Bias²-Variância vs p
ps = list(range(1, 12))
b2s, vs, ms = [], [], []
pp = {p: np.zeros((n_trials, len(x_all))) for p in ps}
for t in range(n_trials):
    x_t = np.sort(np.random.uniform(0, 3, n_pts))
    y_t = f_star(x_t) + np.random.normal(0, sigma, n_pts)
    for p in ps:
        B_te = np.column_stack([(x_all/3)**j for j in range(p+1)])
        th = fit_poly(x_t, y_t, p)
        pp[p][t] = np.clip(B_te @ th, -4, 6)
for p in ps:
    mu_p = pp[p].mean(0)
    b2s.append(float(np.mean((mu_p - f_star(x_all))**2)))
    vs.append(float(np.mean(pp[p].var(0))))
    ms.append(b2s[-1] + vs[-1] + sigma**2)

ax = axes[1, 0]
ax.plot(ps, b2s, 'r-o', ms=5, lw=2, label='Bias²')
ax.plot(ps, vs,  'b-s', ms=5, lw=2, label='Variância')
ax.plot(ps, ms,  'k-^', ms=5, lw=2.5, label='MSE Total')
ax.axhline(sigma**2, color='gray', ls=':', lw=1.5, label=f'Ruído = {sigma**2:.2f}')
best = ps[int(np.argmin(ms))]
ax.axvline(best, color='green', ls='--', lw=2, label=f'p* = {best}')
ax.set_title('Bias²-Variância vs número de bases p', fontweight='bold')
ax.set_xlabel('p'); ax.set_ylabel('Erro'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Learning Curves
X_lc, y_lc = make_regression(n_samples=300, n_features=10, n_informative=6, noise=25, random_state=1)
X_lc = StandardScaler().fit_transform(X_lc)
cenarios = [('Alta Variância\n(λ=1e-8)', Ridge(alpha=1e-8), '#E74C3C'),
            ('Bom Equilíbrio\n(λ=1.0)',   Ridge(alpha=1.0),  '#2ECC71'),
            ('Alto Bias\n(λ=1e4)',         Ridge(alpha=1e4),  '#3498DB')]

for ax_idx, (titulo, modelo, cor) in enumerate(cenarios):
    ax = axes[1, ax_idx]
    sizes = np.linspace(0.1, 0.9, 8)
    tr_m, val_m = [], []
    for frac in sizes:
        n_tr = max(10, int(frac * 200))
        tr_sc, val_sc = [], []
        for seed in range(10):
            idx = np.random.permutation(len(X_lc))
            Xtr, ytr = X_lc[idx[:n_tr]], y_lc[idx[:n_tr]]
            Xv,  yv  = X_lc[idx[n_tr:n_tr+50]], y_lc[idx[n_tr:n_tr+50]]
            if len(Xv) < 5: continue
            modelo.fit(Xtr, ytr)
            tr_sc.append(np.mean((ytr - modelo.predict(Xtr))**2))
            val_sc.append(np.mean((yv - modelo.predict(Xv))**2))
        tr_m.append(np.mean(tr_sc)); val_m.append(np.mean(val_sc))
    ns = [max(10, int(f*200)) for f in sizes]
    ax.plot(ns, tr_m,  'o-', color=cor,    lw=2, label='Erro Treino')
    ax.plot(ns, val_m, 's--', color='gray', lw=2, label='Erro Validação')
    ax.fill_between(ns, tr_m, val_m, alpha=0.15, color=cor)
    ax.set_title(f'Learning Curve — {titulo}', fontweight='bold', fontsize=9)
    ax.set_xlabel('N amostras de treino'); ax.set_ylabel('MSE')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Exercício 2 — Bias-Variância e Diagnóstico', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
---
# EXERCÍCIO 3 — (20 pontos)
**Tema: Regularização e Cross-Validation**

Seja $\mathcal{L}: \mathbb{R}^p \to \mathbb{R}$ uma função de perda e $\mathbf{B} \in \mathbb{R}^{n \times p}$ a matriz de design.

**(a)** Qual é o propósito da regularização? Escreva a forma geral da perda regularizada e explique o papel do hiperparâmetro $\lambda$. O que acontece nos casos extremos $\lambda = 0$ e $\lambda \to \infty$? **(2)**

**(b)** Explique a **regularização de Tikhonov variacional**: escreva o funcional de regularização, a função de perda correspondente, e derive a solução analítica para o caso linear com $\Gamma = I$. Explique por que $(\mathbf{B}^\top\mathbf{B} + \lambda I)$ é sempre invertível para $\lambda > 0$. **(5)**

**(c)** Compare L1 (Lasso) e L2 (Ridge) matematicamente: escreva as penalidades, discuta a existência de solução analítica, e explique geometricamente **por que Lasso cria zeros exatos** nos coeficientes enquanto Ridge não. **(4)**

**(d)** Descreva o **algoritmo de Early Stopping** para descida de gradiente. Explique a equivalência teórica entre early stopping e regularização Ridge (a correspondência $t \leftrightarrow \lambda$). **(4)**

**(e)** Descreva o procedimento completo de **seleção de $\lambda$ via K-Fold CV**. Por que o conjunto de teste nunca deve ser usado neste procedimento? **(3)**

**(f)** O que é o **Elastic Net**? Escreva a função de perda e explique quando é preferível ao Ridge ou ao Lasso isoladamente. **(2)**

## ✅ Resposta — Exercício 3

---

### (a) Propósito da regularização

A regularização previne o overfitting adicionando uma penalidade à função de perda:

$$\boxed{\mathcal{L}_\text{reg}(\boldsymbol{\theta}) = \underbrace{\mathcal{L}_\text{dados}(\boldsymbol{\theta})}_{\text{fidelidade aos dados}} + \lambda \underbrace{\mathcal{R}(\boldsymbol{\theta})}_{\text{penalidade de complexidade}}}$$

**O papel de $\lambda$:**
- $\lambda = 0$: OLS puro — sem penalidade, pode overfit completamente
- $\lambda \to \infty$: penalidade domina → $\boldsymbol{\theta} \to \mathbf{0}$ → modelo trivial (prediz sempre zero) → underfitting
- $\lambda$ ótimo: equilíbrio entre fidelidade e simplicidade, determinado por Cross-Validation

> **Intuição:** $\lambda$ é um "dial de complexidade". Cross-Validation encontra o valor ótimo ao explorar uma grade de valores e selecionar o que minimiza o erro de validação.

---

### (b) Regularização de Tikhonov — derivação completa

O framework de Tikhonov variacional:
$$\hat{f} = \arg\min_{f} \left[\frac{1}{n}\sum_{i=1}^n \ell(f(x_i), y_i) + \lambda \|\Gamma f\|^2\right]$$

**O funcional de regularização $\|\Gamma f\|^2$:**
- $\Gamma = I$: $\|\theta\|^2$ → penaliza parâmetros grandes → **Ridge**
- $\Gamma = D^1$: $\|f'\|^2$ → penaliza inclinação (suaviza)
- $\Gamma = D^2$: $\|f''\|^2$ → penaliza curvatura (splines suavizantes)

**Derivação para $\Gamma = I$, modelo linear:**

$$\min_{\boldsymbol{\theta}} \|\mathbf{B}\boldsymbol{\theta} - \mathbf{y}\|^2 + \lambda\|\boldsymbol{\theta}\|^2$$

Gradiente:
$$\nabla_\theta \mathcal{L}_\text{reg} = -2\mathbf{B}^\top(\mathbf{y} - \mathbf{B}\theta) + 2\lambda\theta = -2\mathbf{B}^\top\mathbf{y} + 2(\mathbf{B}^\top\mathbf{B} + \lambda I)\theta = \mathbf{0}$$

$$\boxed{\boldsymbol{\theta}^*_\text{Ridge} = (\mathbf{B}^\top\mathbf{B} + \lambda I)^{-1}\mathbf{B}^\top\mathbf{y}}$$

**Por que $(\mathbf{B}^\top\mathbf{B} + \lambda I)$ é sempre invertível para $\lambda > 0$?**

Seja $\mu_1 \geq \mu_2 \geq \cdots \geq \mu_p \geq 0$ os autovalores de $\mathbf{B}^\top\mathbf{B}$ (sempre não-negativos pois é semidefinida positiva).

Os autovalores de $(\mathbf{B}^\top\mathbf{B} + \lambda I)$ são $\mu_j + \lambda$. Para $\lambda > 0$:
$$\mu_j + \lambda \geq 0 + \lambda = \lambda > 0 \quad \forall j$$

Todos os autovalores são estritamente positivos → a matriz é **definida positiva** → sempre invertível.

---

### (c) L1 vs L2 — diferença e geometria

| Propriedade | L2 (Ridge) | L1 (Lasso) |
|-------------|-----------|------------|
| Penalidade | $\lambda\|\theta\|^2 = \lambda\sum_j \theta_j^2$ | $\lambda\|\theta\|_1 = \lambda\sum_j |\theta_j|$ |
| Região de restrição | **Bola esférica** (lisa, sem cantos) | **Diamante/octaedro** (vértices nos eixos) |
| Solução analítica | **Sim** | Não (subgradiente) |
| Coeficientes | Encolhidos, nunca zero exato | Muitos **exatamente zero** |

**Por que Lasso cria zeros exatos (geometria):**

Na formulação constrained $\min_\theta \mathcal{L}(\theta)$ s.a. $\|\theta\|_1 \leq r$:
- As curvas de nível de $\mathcal{L}$ são elipses centradas na solução OLS
- O diamante L1 tem **vértices nos eixos coordenados** (onde $\theta_j = 0$)
- A solução é o ponto onde a elipse "toca" o diamante pela primeira vez
- Geometricamente, as elipses quase sempre atingem primeiro um vértice do diamante — onde exatamente $\theta_j = 0$

Para L2 (bola esférica e lisa): a tangência ocorre no interior da bola, nunca num eixo → coeficientes nunca exatamente zero.

---

### (d) Early Stopping e equivalência com Ridge

**O algoritmo de Early Stopping:**
```
Inicialize θ⁰ = 0 (ou aleatoriamente)
Para t = 1, 2, 3, …:
    θᵗ = θᵗ⁻¹ - η · ∇θ L(θᵗ⁻¹)   [passo de descida de gradiente]
    Calcule err_val(θᵗ) no conjunto de validação
    Se err_val começa a crescer: PARE e retorne θᵗ
```

**A equivalência com Ridge:**

Pelo gradient descent no MSE com learning rate $\eta$ pequeno, após $t$ iterações (com $\theta^0 = 0$):
$$\theta^t \approx (\mathbf{B}^\top\mathbf{B} + \lambda_t I)^{-1}\mathbf{B}^\top\mathbf{y}, \quad \lambda_t \approx \frac{1}{\eta \cdot t}$$

**A correspondência é:** $t \uparrow$ (mais iterações) $\leftrightarrow$ $\lambda \downarrow$ (menos regularização).
- $t$ pequeno (early stop) → $\lambda$ grande → modelo simples, mais regularizado
- $t$ grande (treinamento completo) → $\lambda \to 0$ → OLS puro, pode overfit

> **Implicação prática:** Parar cedo é equivalente a adicionar regularização Ridge sem precisar resolver o sistema linear. Em redes neurais (onde não existe solução Ridge analítica), early stopping é o método de regularização mais popular.

---

### (e) Seleção de $\lambda$ via K-Fold CV

**Procedimento completo:**

```
1. Defina uma grade de candidatos: Λ = {λ₁, λ₂, …, λₘ} (ex: logspace de 1e-4 a 1e4)
2. Divida o dataset de treino em K folds iguais (ex: K=5)
3. Para cada λ ∈ Λ:
     Para k = 1, …, K:
         Treine o modelo com λ em todos os folds exceto k: f_{-k}
         Avalie o erro no fold k: err_k = L(f_{-k}, D_k)
     CV_err(λ) = (1/K) Σₖ err_k
4. Selecione λ* = argmin_λ CV_err(λ)   [ou aplique a regra 1-SE]
5. Retreine o modelo FINAL com λ* usando TODO o dataset de treino
6. Reporte o erro final usando o conjunto de TESTE (usado apenas uma vez)
```

**Por que o conjunto de teste nunca deve ser usado para selecionar $\lambda$?**

Selecionar $\lambda$ usando o erro de teste é **data snooping** (vazamento de informação). O conjunto de teste deveria representar dados "do futuro" — nunca vistos. Se o escolhermos baseados nele, estamos implicitamente ajustando $\lambda$ ao teste, tornando o erro de teste um estimador otimista e não mais um estimador válido de generalização. O modelo pareceria melhor do que realmente é para dados genuinamente novos.

---

### (f) Elastic Net

O **Elastic Net** combina penalidades L1 e L2:

$$\mathcal{L}_\text{EN}(\theta) = \|\mathbf{B}\theta - y\|^2 + \lambda_1\|\theta\|_1 + \lambda_2\|\theta\|^2$$

Alternativamente escrito com $\lambda$ total e mistura $\alpha \in [0,1]$:
$$\mathcal{L}_\text{EN}(\theta) = \|\mathbf{B}\theta - y\|^2 + \lambda\left[\alpha\|\theta\|_1 + (1-\alpha)\|\theta\|^2\right]$$

**Quando usar:**
- Quando há **muitas features correlacionadas**: Lasso tende a selecionar apenas uma por grupo (aleatoriamente), Ridge inclui todas mas não elimina nenhuma. Elastic Net inclui grupos inteiros de features correlacionadas enquanto ainda elimina as irrelevantes.
- Quando há mais features do que amostras ($p \gg n$): Lasso puro seleciona no máximo $n$ features; Elastic Net pode selecionar mais.
- Na prática: quando você quer esparsidade (como Lasso) mas com mais estabilidade numérica (como Ridge).

In [ ]:
# === Visualização — Exercício 3: Ridge vs Lasso, Regularization Paths, Geometria ===

np.random.seed(7)
n, p_feat = 40, 15
X_reg, y_reg = make_regression(n_samples=n, n_features=p_feat, n_informative=5,
                                noise=15, random_state=7)
X_reg = StandardScaler().fit_transform(X_reg)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Esparsidade: Ridge vs Lasso
ax = axes[0]
ridge = Ridge(alpha=1.0).fit(X_reg, y_reg)
lasso = Lasso(alpha=0.5, max_iter=10000).fit(X_reg, y_reg)
idx = np.arange(p_feat)
ax.bar(idx-0.2, ridge.coef_, 0.4, color='#3498DB', alpha=0.8,
       label=f'Ridge  ({np.sum(np.abs(ridge.coef_)<0.001)} zeros)')
ax.bar(idx+0.2, lasso.coef_, 0.4, color='#E74C3C', alpha=0.8,
       label=f'Lasso  ({np.sum(lasso.coef_==0)} zeros exatos)')
ax.axhline(0, color='k', lw=0.8)
ax.set_title('L2 vs L1: Esparsidade dos Coeficientes\n(Lasso força $\\theta_j=0$ exatamente)', fontweight='bold')
ax.set_xlabel('Feature j'); ax.set_ylabel('$\\theta_j$')
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')

# Caminhos de regularização Ridge
ax = axes[1]
alphas = np.logspace(-3, 3, 80)
coefs_r = [Ridge(alpha=a).fit(X_reg, y_reg).coef_ for a in alphas]
coefs_r = np.array(coefs_r)
for j in range(p_feat):
    ax.semilogx(alphas, coefs_r[:, j], lw=1.5, alpha=0.7)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_title('Caminho de Regularização Ridge\n(todos os coeficientes → 0 suavemente)', fontweight='bold')
ax.set_xlabel('λ'); ax.set_ylabel('Coeficiente $\\theta_j$')
ax.grid(alpha=0.3)
ax.text(0.02, 0.95, 'λ→0: OLS', transform=ax.transAxes, va='top', fontsize=9, color='#E74C3C')
ax.text(0.72, 0.95, 'λ→∞: θ≈0', transform=ax.transAxes, va='top', fontsize=9, color='#2ECC71')

# Geometria: região de restrição L1 vs L2
ax = axes[2]
theta = np.linspace(-1.5, 1.5, 400)
T1, T2 = np.meshgrid(theta, theta)
# Perda quadrática centrada em (0.9, 0.6)
L = (T1-0.9)**2 + (T2-0.6)**2
ax.contour(T1, T2, L, levels=10, cmap='RdYlGn', alpha=0.6)
# Bola L2
r = 0.7
ang = np.linspace(0, 2*np.pi, 300)
ax.plot(r*np.cos(ang), r*np.sin(ang), 'b-', lw=2.5, label='L2: $\\|\\theta\\|^2 \\leq r$')
ax.fill(r*np.cos(ang), r*np.sin(ang), alpha=0.1, color='blue')
# Bola L1 (diamante)
diamond_x = [0, r, 0, -r, 0]
diamond_y = [r, 0, -r, 0, r]
ax.plot(diamond_x, diamond_y, 'r-', lw=2.5, label='L1: $\\|\\theta\\|_1 \\leq r$')
ax.fill(diamond_x, diamond_y, alpha=0.1, color='red')
# Ponto de contato L2 (sem esparsidade)
ax.scatter([0.52], [0.45], s=120, color='blue', zorder=6, marker='*',
           label='Solução Ridge (interior suave)')
# Ponto de contato L1 (esparsidade no vértice)
ax.scatter([r], [0], s=120, color='red', zorder=6, marker='*',
           label='Solução Lasso (vértice → $\\theta_2=0$)')
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_title('Geometria L1 vs L2\n(por que Lasso gera esparsidade)', fontweight='bold')
ax.set_xlabel('$\\theta_1$'); ax.set_ylabel('$\\theta_2$')
ax.legend(fontsize=8, loc='upper left'); ax.grid(alpha=0.3)
ax.set_xlim(-1.2, 1.4); ax.set_ylim(-1.0, 1.2)

plt.suptitle('Exercício 3 — Regularização: Tikhonov (Ridge) e Lasso (L1)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
---
# EXERCÍCIO 4 — (20 pontos)
**Tema: Visão Probabilística e Regularização Bayesiana**

**(a)** Mostre como o MSE emerge **naturalmente** da hipótese de ruído Gaussiano $\varepsilon_i \sim \mathcal{N}(0,\sigma^2)$. Escreva o modelo, a likelihood, o log-likelihood negativo e identifique a equivalência com o MSE. **(4)**

**(b)** Explique a conexão Bayesiana entre o tipo de **prior** sobre $\boldsymbol{\theta}$ e o tipo de regularização resultante. Derive o estimador MAP para o Prior Gaussiano e para o Prior de Laplace, e mostre que correspondem a Ridge e Lasso respectivamente. **(5)**

**(c)** Compare **MLE e MAP**: o que muda na formulação? Em que condição MAP = MLE? Qual a interpretação de cada um em termos de regularização? **(4)**

**(d)** Qual é a **interpretação Bayesiana do hiperparâmetro $\lambda$**? Como $\lambda$ se relaciona com a relação $\sigma^2_\varepsilon / \tau^2$ (ruído nos dados vs. variância do prior)? **(3)**

**(e)** Dê a **definição formal de convexidade** de uma função. Prove que o MSE $\mathcal{L}(\theta) = \|\mathbf{B}\theta - y\|^2$ é convexo em $\theta$. Qual é a consequência fundamental para algoritmos de otimização? **(4)**

## ✅ Resposta — Exercício 4

---

### (a) MSE emerge do ruído Gaussiano

**Hipótese:** $y_i = f_\theta(x_i) + \varepsilon_i$, com $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$ i.i.d.

**Passo 1 — Distribuição condicional:**
$$y_i \mid x_i, \theta \sim \mathcal{N}(f_\theta(x_i), \sigma^2) \implies p(y_i \mid x_i, \theta) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\left(-\frac{(y_i - f_\theta(x_i))^2}{2\sigma^2}\right)$$

**Passo 2 — Likelihood conjunta (dados i.i.d.):**
$$p(D \mid \theta) = \prod_{i=1}^n p(y_i \mid x_i, \theta)$$

**Passo 3 — Log-Likelihood negativo:**
$$-\log p(D \mid \theta) = \sum_{i=1}^n \frac{(y_i - f_\theta(x_i))^2}{2\sigma^2} + \underbrace{\frac{n}{2}\log(2\pi\sigma^2)}_{\text{constante em }\theta}$$

**Passo 4 — Identificação:**
$$\hat{\theta}_\text{MLE} = \arg\min_\theta \left[-\log p(D|\theta)\right] = \arg\min_\theta \sum_{i=1}^n (y_i - f_\theta(x_i))^2 = \arg\min_\theta \text{MSE}(\theta)$$

> **Conclusão:** Minimizar o MSE **é exatamente** maximizar a likelihood Gaussiana. O MSE não é arbitrário — ele emerge naturalmente da suposição de ruído Gaussiano. Se o ruído fosse Laplaciano, minimizaríamos o MAE $\sum |y_i - \hat{y}_i|$.

---

### (b) Prior → Regularização (conexão Bayesiana)

**Teorema de Bayes:** $p(\theta \mid D) \propto p(D \mid \theta) \cdot p(\theta)$

O estimador **MAP** minimiza $-\log p(\theta \mid D) = -\log p(D|\theta) - \log p(\theta)$:
$$\hat{\theta}_\text{MAP} = \arg\min_\theta \left[\underbrace{\text{MSE}(\theta)}_{\text{= NLL Gaussiano}} + \underbrace{(-\log p(\theta))}_{\text{= penalidade}}\right]$$

**Prior Gaussiano** $p(\theta) = \mathcal{N}(0, \tau^2 I)$:
$$-\log p(\theta) = \frac{\|\theta\|^2}{2\tau^2} + \text{const} \implies \hat{\theta}_\text{MAP} = \arg\min_\theta \left[\text{MSE} + \underbrace{\frac{\sigma^2}{\tau^2}}_{\lambda}\|\theta\|^2\right] \quad\Longleftrightarrow\quad \textbf{Ridge (L2)}$$

O prior Gaussiano crê que coeficientes são pequenos e distribuídos simetricamente — coeficientes grandes são improváveis. Ridge penaliza coeficientes grandes.

**Prior de Laplace** $p(\theta_j) = \frac{1}{2b}\exp(-|\theta_j|/b)$:
$$-\log p(\theta) = \frac{\|\theta\|_1}{b} + \text{const} \implies \hat{\theta}_\text{MAP} = \arg\min_\theta \left[\text{MSE} + \underbrace{\frac{\sigma^2}{b}}_{\lambda}\|\theta\|_1\right] \quad\Longleftrightarrow\quad \textbf{Lasso (L1)}$$

O prior de Laplace tem cúspide em zero e caudas pesadas — crê que muitos coeficientes são exatamente zero (esparsidade) e alguns poucos são grandes.

| Prior | $-\log p(\theta)$ | Regularização | $\lambda$ |
|---|---|---|---|
| $\mathcal{N}(0, \tau^2 I)$ | $\frac{1}{2\tau^2}\|\theta\|^2$ | L2 Ridge | $\sigma^2/\tau^2$ |
| $\text{Lap}(0, b)$ | $\frac{1}{b}\|\theta\|_1$ | L1 Lasso | $\sigma^2/b$ |

---

### (c) MLE vs MAP

| | MLE | MAP |
|---|---|---|
| **Formulação** | $\arg\max_\theta p(D\|\theta)$ | $\arg\max_\theta p(\theta\|D) = \arg\max_\theta [p(D\|\theta)\cdot p(\theta)]$ |
| **Equivalente a** | Minimizar MSE | Minimizar MSE + penalidade |
| **Interpretação** | Máxima verossimilhança; sem informação prévia | Máxima posteriori; incorpora conhecimento prévio sobre $\theta$ |
| **Regularização** | Nenhuma (ou prior uniforme implícito) | Ridge (prior Gaussiano) ou Lasso (prior Laplace) |

**Quando MAP = MLE?**

MAP = MLE quando o prior é **uniforme** (não informativo): $p(\theta) = \text{const}$ para todo $\theta$.
Então $-\log p(\theta) = \text{const}$ → não muda o argmin → MAP e MLE coincidem.

Isso equivale a $\lambda = 0$ (sem regularização) ou $\tau \to \infty$ (variância do prior infinita — crença completamente vaga).

---

### (d) Interpretação Bayesiana de $\lambda$

Da derivação MAP com prior Gaussiano, vimos que:
$$\lambda = \frac{\sigma^2_\varepsilon}{\tau^2}$$

onde $\sigma^2_\varepsilon$ é a variância do ruído nos dados e $\tau^2$ é a variância do prior sobre $\theta$.

**Interpretação:**
- **$\sigma^2_\varepsilon$ grande** (dados muito ruidosos) → $\lambda$ grande → regularização forte: quando os dados são pouco confiáveis, confiamos mais no prior (coeficientes pequenos)
- **$\tau^2$ grande** (prior vago, "não sei nada sobre $\theta$") → $\lambda$ pequeno → regularização fraca: deixamos os dados falarem mais
- **$\tau^2$ pequeno** (prior forte, "acredito que $\theta$ é pequeno") → $\lambda$ grande → regularização forte: o prior domina

> **Resumo:** $\lambda$ representa a razão entre a incerteza nos dados e a incerteza no prior. Um $\lambda$ grande significa "os dados são ruidosos *e/ou* acredito fortemente que os parâmetros são pequenos".

---

### (e) Convexidade — definição, prova e consequência

**Definição:** $\mathcal{L}: \mathbb{R}^p \to \mathbb{R}$ é convexa se $\forall\, \theta_1, \theta_2 \in \mathbb{R}^p$, $\forall\, \alpha \in [0,1]$:
$$\mathcal{L}(\alpha\theta_1 + (1-\alpha)\theta_2) \leq \alpha\mathcal{L}(\theta_1) + (1-\alpha)\mathcal{L}(\theta_2)$$

Geometricamente: o gráfico fica abaixo de qualquer corda (segmento de reta entre dois pontos) — função "côncava para cima como uma bacia".

**Prova de que o MSE é convexo:**

$$\mathcal{L}(\theta) = \|\mathbf{B}\theta - y\|^2 = \theta^\top\underbrace{\mathbf{B}^\top\mathbf{B}}_{\mathbf{M}}\theta - 2y^\top\mathbf{B}\theta + y^\top y$$

Esta é uma função quadrática. A Hessiana é $\nabla^2_\theta \mathcal{L} = 2\mathbf{B}^\top\mathbf{B}$. Como $\mathbf{B}^\top\mathbf{B}$ é **semidefinida positiva** (para qualquer vetor $v$: $v^\top\mathbf{B}^\top\mathbf{B}v = \|\mathbf{B}v\|^2 \geq 0$), a Hessiana é semidefinida positiva → $\mathcal{L}$ é convexa.

**Consequência fundamental:**

> Se $\mathcal{L}$ é convexa → todo minimizador local é também global.
> Se $\mathcal{L}$ é estritamente convexa → o minimizador global é **único**.

Algoritmos como descida de gradiente (GD), SGD, Adam, L-BFGS são **garantidos** de encontrar o ótimo global para funções convexas — sem risco de convergir para um mínimo local ruim. Isso não vale para redes neurais profundas (não-convexas).

In [ ]:
# === Visualização — Exercício 4: Prior Bayesiano e Convexidade ===

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Prior Gaussiano vs Laplace
ax = axes[0]
theta_v = np.linspace(-4, 4, 400)
p_gauss  = stats.norm(0, 1).pdf(theta_v)
p_lap    = stats.laplace(0, 1).pdf(theta_v)
ax.plot(theta_v, p_gauss, 'b-', lw=2.5, label='Prior Gaussiano → L2 (Ridge)')
ax.plot(theta_v, p_lap,   'r--', lw=2.5, label='Prior Laplace → L1 (Lasso)')
ax.fill_between(theta_v, p_gauss, alpha=0.1, color='blue')
ax.fill_between(theta_v, p_lap,   alpha=0.1, color='red')
ax.annotate('Cúspide em 0\n→ Esparsidade!', (0, 0.5), (1.2, 0.45),
            arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=9)
ax.set_title('Prior $p(\\theta)$: Gaussiano vs Laplace\n'
             '(conexão com tipo de regularização)', fontweight='bold')
ax.set_xlabel('θ'); ax.set_ylabel('p(θ)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# -log Prior = Penalidade
ax = axes[1]
neg_lg = -np.log(p_gauss + 1e-12)
neg_ll = -np.log(p_lap + 1e-12)
neg_lg -= neg_lg.min(); neg_ll -= neg_ll.min()
ax.plot(theta_v, neg_lg, 'b-',  lw=2.5, label='$-\\log$ Gaussiano $= \\|\\theta\\|^2$ (L2)')
ax.plot(theta_v, neg_ll, 'r--', lw=2.5, label='$-\\log$ Laplace $= \\|\\theta\\|_1$ (L1)')
ax.set_ylim(0, 9)
ax.set_title('-log Prior = Penalidade de Regularização\n'
             'MAP = minimizar MSE + λ·Penalidade', fontweight='bold')
ax.set_xlabel('θ'); ax.set_ylabel('-log p(θ) = penalidade')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Convexidade: convexa vs não-convexa
ax = axes[2]
x_cv = np.linspace(-3, 3.5, 300)
f_cvx  = x_cv**2 + 0.5
f_ncvx = x_cv**4 - 3.5*x_cv**2 + x_cv + 3
ax.plot(x_cv, f_cvx,  'b-', lw=2.5, label='Convexa: $f(x)=x^2+0.5$')
ax.scatter([0], [0.5], s=150, color='blue', zorder=6, marker='*')
ax.text(0.1, 0.8, 'Global = Local', color='blue', fontsize=8)
ax.plot(x_cv, f_ncvx, 'r-', lw=2.5, label='Não-convexa: $x^4-3.5x^2+x+3$')
for lx, label_t in [(-1.3, 'Local\nmin'), (1.35, 'Global\nmin')]:
    lv = lx**4 - 3.5*lx**2 + lx + 3
    ax.scatter([lx], [lv], s=100, color='red', zorder=6)
    ax.annotate(label_t, (lx, lv), (lx-0.8, lv+1.2),
                arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=8)
ax.set_title('Convexidade:\nLocal min = Global min (convexa)', fontweight='bold')
ax.set_xlabel('θ'); ax.set_ylabel('L(θ)')
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_ylim(-1, 12)

plt.suptitle('Exercício 4 — Visão Probabilística e Regularização Bayesiana',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
---
# EXERCÍCIO BÔNUS — (10 pontos)
**Tema: Bootstrap e Cross-Validation Avançado**

**(a)** Descreva o **algoritmo do Bootstrap não-paramétrico** passo a passo. Explique intuitivamente por que reamostrar com reposição simula a coleta de novos datasets. Como é usado para construir Intervalos de Confiança? Mencione ao menos 3 tipos de IC bootstrap. **(3)**

**(b)** Compare os seguintes métodos de Cross-Validation: **Hold-Out**, **K-Fold**, **Stratified K-Fold** e **Leave-One-Out (LOO)**. Para cada um, descreva o mecanismo, cite a vantagem principal, a desvantagem principal, e indique quando usar. **(4)**

**(c)** Prove matematicamente que **Bagging** reduz a variância. Use a expressão $\text{Var}(\bar{X}) = \text{Var}(X)/B$ para estimadores independentes e explique por que na prática os modelos bootstrap **não são completamente independentes** (correlação positiva). Qual é a consequência para a redução real de variância? **(3)**

## ✅ Resposta — Exercício Bônus

---

### (a) Bootstrap Não-Paramétrico e Intervalos de Confiança

**O problema:** Como quantificar a incerteza de um estimador quando não há fórmula analítica para a sua distribuição amostral? (ex: IC para o coeficiente Ridge, AUC de um classificador, correlação de Spearman)

**O "truque" do Bootstrap:** O dataset $D = \{z_1, \ldots, z_n\}$ é tratado como a melhor aproximação disponível da população. Reamostrar *com reposição* simula o processo de coletar novos datasets dessa população.

> **Intuição:** Imagine uma urna com $n$ bolas (seu dataset). Para estimar a variabilidade de uma estatística, sorteie $n$ bolas *com reposição* (algumas sairão 2-3x, outras não sairão), calcule a estatística, e repita $B$ vezes. A variabilidade das $B$ estatísticas bootstrap estima a variabilidade real.

**Por que com reposição?** Sem reposição, cada amostra seria uma permutação do dataset original — sempre o mesmo conjunto. Com reposição, cada amostra $D^b$ é genuinamente diferente, simulando um novo dataset da mesma população.

**Algoritmo:**
```
Entrada: dados D = {z₁, …, zₙ}, estatística S(·), B replicatas

Para b = 1, …, B:
    1. Amostra D^b: n obs. de D COM REPOSIÇÃO
       (em média: 63.2% das obs. originais únicas aparecem)
    2. s^b = S(D^b)

Saída: {s¹, …, s^B}, erro padrão = std({s^b})
```

**Três tipos de IC Bootstrap (95%):**

| Tipo | Fórmula | Quando usar |
|------|---------|-------------|
| **Normal** | $\hat{s} \pm 1.96 \cdot \text{std}(\{s^b\})$ | Distribuição bootstrap aproximadamente Normal |
| **Percentil** | $[q_{2.5\%}(\{s^b\}),\; q_{97.5\%}(\{s^b\})]$ | Distribuição assimétrica; mais robusto |
| **BCa (Bias-Corrected Accelerated)** | Ajusta $q_\alpha$ para viés e aceleração | Distribuição assimétrica/enviesada; mais preciso |

---

### (b) Métodos de Cross-Validation — Comparação

**A fórmula geral K-Fold:**
$$\hat{\text{Err}} = \frac{1}{K}\sum_{k=1}^K \mathcal{L}(f_{-k}, D_k)$$

| Método | Mecanismo | Vantagem | Desvantagem | Quando usar |
|--------|-----------|----------|-------------|-------------|
| **Hold-Out** | Split único (ex: 75/25); 1 treino, 1 avaliação | Extremamente rápido: $O(1)$ | Alta variabilidade: depende do split específico; pode ser enganoso com $N$ pequeno | $N > 50.000$; estimativas preliminares rápidas |
| **K-Fold** (K=5 ou 10) | K folds; cada um é validação uma vez; média dos erros | Usa todos os dados; muito menos variável que Hold-Out; bom equilíbrio | $K\times$ mais lento que Hold-Out | **Padrão para a maioria dos problemas** |
| **Stratified K-Fold** | K-Fold preservando proporção de classes em cada fold | Robusto a desbalanceamento de classes | Ligeiramente mais complexo de implementar | **Sempre em classificação** com classes raras ou desbalanceadas |
| **LOO** ($K=N$) | Cada obs. é validação uma vez; $N$ modelos treinados | Mínimo viés (treina com $N-1$); determinístico | $O(N)$ treinos; alta variância do estimador | $N$ muito pequeno ($< 50$–100) |

> **Regra de ouro:** Para classificação → **Stratified K-Fold**. Para séries temporais → **TimeSeriesSplit** (nunca K-Fold padrão — vazaria o futuro!). Para $N$ pequeno → LOO. Para $N$ enorme → Hold-Out.

---

### (c) Bagging — prova de redução de variância e limitação prática

**Caso ideal (estimadores independentes):**

Sejam $\hat{f}^{(1)}, \ldots, \hat{f}^{(B)}$ estimadores i.i.d. com $\mathbb{E}[\hat{f}^{(b)}(x)] = \mu$ e $\text{Var}(\hat{f}^{(b)}(x)) = \sigma^2$.

$$\hat{f}_\text{bag}(x) = \frac{1}{B}\sum_{b=1}^B \hat{f}^{(b)}(x)$$

$$\text{Var}(\hat{f}_\text{bag}) = \text{Var}\!\left(\frac{1}{B}\sum_b \hat{f}^{(b)}\right) \stackrel{\text{independência}}{=} \frac{1}{B^2}\sum_b \text{Var}(\hat{f}^{(b)}) = \frac{1}{B^2} \cdot B \cdot \sigma^2 = \frac{\sigma^2}{B}$$

Redução: $\text{Var}(\hat{f}_\text{bag}) = \sigma^2 / B$ — fator de $B$.

**O problema na prática: correlação positiva entre modelos bootstrap**

Os $B$ modelos bootstrap não são completamente independentes — todos são treinados no *mesmo dataset* original $D$ (com perturbações via resampling). Dois modelos $\hat{f}^{(b)}$ e $\hat{f}^{(b')}$ treinados em $D$ têm correlação positiva $\rho > 0$ porque "viram" os mesmos dados originais.

Quando os estimadores têm correlação $\rho$:
$$\text{Var}(\hat{f}_\text{bag}) = \rho \sigma^2 + \frac{(1-\rho)\sigma^2}{B}$$

**Consequências:**
- Para $B \to \infty$: $\text{Var}(\hat{f}_\text{bag}) \to \rho\sigma^2$ — a variância não vai a zero, mas é limitada por $\rho\sigma^2$
- A correlação $\rho$ é o "teto" da redução de variância que Bagging pode atingir
- **Random Forest** resolve isso: além do resampling de dados, também reamostra quais features estão disponíveis em cada nó da árvore → os modelos ficam menos correlacionados → $\rho$ menor → maior redução de variância

---
---
# EXERCÍCIO 5 — (10 pontos)
**Tema: Questões Aplicadas e Conceituais (todos os tópicos)**

**(a)** Você tem $n = 50$ amostras e $p = 200$ features. A Equação Normal falha. Que regularização escolheria e por quê? Escreva a solução resultante. **(3)**

**(b)** Um modelo apresenta **erro de treino $= 0{,}02$** e **erro de validação $= 0{,}87$**. Identifique o problema, a causa e proponha **duas estratégias** para corrigi-lo. **(3)**

**(c)** Por que **nunca** devemos usar o conjunto de teste para selecionar $\lambda$? O que é *data snooping* e qual a consequência prática? **(2)**

**(d)** Um colega propõe Bootstrap em vez de Cross-Validation para estimar o erro de generalização. Compare os dois: vantagem, desvantagem e quando cada um é preferível. **(2)**

## ✅ Resposta — Exercício 5

---

### (a) $n=50$, $p=200$: qual regularização?

Com $p > n$, a matriz $\mathbf{B}^\top\mathbf{B}$ tem posto $\leq 50$ → singular → sem solução única.

**Lasso (L1):** faz seleção automática de features (zeros exatos) — ideal quando muitas das 200 features são irrelevantes. Solução: $\hat{\theta}_\text{Lasso} = \arg\min_\theta \|\mathbf{B}\theta - y\|^2 + \lambda\|\theta\|_1$

**Ridge:** alternativa se não há razão para esparsidade: $\hat{\theta} = (\mathbf{B}^\top\mathbf{B} + \lambda I)^{-1}\mathbf{B}^\top y$ — sempre invertível para $\lambda > 0$, pois $\mu_j + \lambda > 0$.

---

### (b) Treino 0,02 — Validação 0,87

**Diagnóstico: Alta Variância (Overfitting grave).** O modelo memorizou os dados de treino incluindo o ruído.

**Estratégia 1 — Aumentar $\lambda$:** força coeficientes menores → menor variância. Selecionar $\lambda^*$ via K-Fold CV.

**Estratégia 2 — Reduzir $p$:** usar menos funções de base ou Lasso para eliminar features irrelevantes.

---

### (c) Data Snooping

Usar o conjunto de teste para escolher $\lambda$ contamina a avaliação: o erro de teste seria otimista (inflado pela seleção) e não representaria dados genuinamente novos.

**Regra:** Use CV no treino para selecionar $\lambda$. Teste usado **uma única vez** no final.

---

### (d) Bootstrap vs Cross-Validation

| Aspecto | K-Fold CV | Bootstrap |
|---------|-----------|----------|
| Viés do estimador de erro | Baixo | Alto (pessimista: treina em ~63% dos dados) |
| Seleção de modelo | **Padrão recomendado** | Não ideal |
| Ponto forte | Estimativa de generalização | ICs para estimadores (coeficientes, AUC) |

Use **CV para seleção de modelo** e **bootstrap para quantificar incerteza de estimadores**.

---
---
# EXERCÍCIO 6 — (10 pontos)
**Tema: L1 Avançado — Projeção, $R^2$ e Normalização de Features**

**(a)** Defina a **Hat Matrix** $\mathbf{H} = \mathbf{B}(\mathbf{B}^\top\mathbf{B})^{-1}\mathbf{B}^\top$. O que ela faz geometricamente? Mostre que os resíduos $\hat{\mathbf{e}} = \mathbf{y} - \hat{\mathbf{y}}$ são ortogonais às predições $\hat{\mathbf{y}}$. **(3)**

**(b)** Defina $R^2 = 1 - SS_{\text{res}}/SS_{\text{tot}}$. Por que $R^2$ **sempre cresce** ao adicionar mais funções de base $p$, mesmo que irrelevantes? Como o **$R^2$ ajustado** corrige esse problema? **(3)**

**(c)** Feature A tem valores em $[0, 10000]$ e Feature B em $[0, 1]$. Como isso afeta a solução Ridge? Qual é o procedimento de **normalização** correto antes de aplicar regularização e por que é essencial? **(4)**

## ✅ Resposta — Exercício 6

---

### (a) Hat Matrix e Projeção Geométrica

$$\mathbf{H} = \mathbf{B}(\mathbf{B}^\top\mathbf{B})^{-1}\mathbf{B}^\top \in \mathbb{R}^{n \times n}$$

$\mathbf{H}$ é o **operador de projeção ortogonal** no espaço coluna de $\mathbf{B}$. A predição $\hat{\mathbf{y}} = \mathbf{H}\mathbf{y}$ é a projeção de $\mathbf{y}$ no subespaço gerado pelas colunas de $\mathbf{B}$ — o ponto mais próximo de $\mathbf{y}$ naquele subespaço.

**$\mathbf{H}$ é idempotente e simétrica:** $\mathbf{H}^2 = \mathbf{H}$, $\mathbf{H}^\top = \mathbf{H}$.

**Ortogonalidade dos resíduos:**

$$\hat{\mathbf{e}}^\top\hat{\mathbf{y}} = \mathbf{y}^\top(I-\mathbf{H})^\top\mathbf{H}\mathbf{y} = \mathbf{y}^\top(\mathbf{H} - \mathbf{H}^2)\mathbf{y} = \mathbf{y}^\top(\mathbf{H} - \mathbf{H})\mathbf{y} = 0$$

Os resíduos estão no complemento ortogonal do espaço coluna de $\mathbf{B}$ — são a componente de $\mathbf{y}$ que o modelo não consegue explicar.

---

### (b) $R^2$ e $R^2$ Ajustado

$$R^2 = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}} = 1 - \frac{\|\mathbf{y} - \hat{\mathbf{y}}\|^2}{\|\mathbf{y} - \bar{y}\mathbf{1}\|^2}$$

**Por que $R^2$ sempre cresce com $p$?** Adicionar qualquer feature expande o espaço coluna de $\mathbf{B}$. A projeção num espaço maior nunca é pior: $SS_{\text{res}}(p+1) \leq SS_{\text{res}}(p)$. No pior caso o modelo ignora a nova feature (coeficiente = 0). Logo $R^2$ nunca decresce — mesmo para features de puro ruído.

**Problema:** Um modelo com $p = n$ terá $R^2 = 1$ (interpola perfeitamente), mas generaliza horrivelmente.

**$R^2$ Ajustado:** Penaliza o número de parâmetros:

$$R^2_{\text{adj}} = 1 - \frac{SS_{\text{res}}/(n-p)}{SS_{\text{tot}}/(n-1)} = 1 - (1-R^2)\frac{n-1}{n-p}$$

O divisor $(n-p)$ penaliza modelos com muitos parâmetros. Se uma feature não melhora suficientemente $SS_{\text{res}}$, o $R^2_{\text{adj}}$ pode cair — sinal correto.

---

### (c) Efeito de Escala e Normalização

**Problema:** Ridge penaliza $\lambda\sum_j \theta_j^2$ com o **mesmo** $\lambda$ para todos.

Feature A em [0, 10000]: $\theta_A$ intrinsecamente pequeno → penalizado pouco.
Feature B em [0, 1]: $\theta_B$ grande para o mesmo efeito → penalizado mais.

A regularização não é equitativa entre features de escalas diferentes — o resultado depende das unidades, não da importância real das features.

**Solução — Padronização (Z-score) antes de regularizar:**

$$\tilde{x}_j = \frac{x_j - \mu_j}{\sigma_j}$$

Após padronização: todas as features têm média 0 e desvio padrão 1. Os coeficientes $\theta_j$ têm interpretação comparável. A penalidade Ridge $\lambda\sum_j \theta_j^2$ aplica-se de forma equitativa.

> **Atenção crítica:** A padronização deve ser calculada **apenas no conjunto de treino** e aplicada ao teste com os mesmos parâmetros ($\mu_j, \sigma_j$) — recalcular no teste seria data leakage.

---
---
# EXERCÍCIO 7 — (10 pontos)
**Tema: L2 Avançado — AIC, BIC e Graus de Liberdade Efetivos**

**(a)** Escreva as fórmulas do **AIC** e do **BIC**. Em que diferem? Quando BIC penaliza mais fortemente do que AIC? **(3)**

**(b)** Defina os **graus de liberdade efetivos** de Ridge: $df(\lambda) = \operatorname{tr}(\mathbf{H}_\lambda)$ onde $\mathbf{H}_\lambda = \mathbf{B}(\mathbf{B}^\top\mathbf{B}+\lambda I)^{-1}\mathbf{B}^\top$. Mostre que $df(\lambda) = \sum_{j=1}^p \frac{\mu_j}{\mu_j + \lambda}$ onde $\mu_j$ são os autovalores de $\mathbf{B}^\top\mathbf{B}$. Interprete o que acontece quando $\lambda \to 0$ e $\lambda \to \infty$. **(4)**

**(c)** Explique qualitativamente o fenômeno do **"double descent"** (dupla descida). Por que o erro de teste pode *cair* novamente após cruzar o ponto de interpolação $p = n$? **(3)**

## ✅ Resposta — Exercício 7

---

### (a) AIC e BIC

$$\text{AIC} = -2\log p(D|\hat{\theta}_\text{MLE}) + 2p$$

$$\text{BIC} = -2\log p(D|\hat{\theta}_\text{MLE}) + p\log n$$

**Diferença:** A penalidade BIC por parâmetro é $\log n$ vs $2$ do AIC.

**Quando BIC penaliza mais?** Sempre que $\log n > 2$, i.e., $n > e^2 \approx 7.4$ — praticamente sempre na prática.

**Consequências:**
- AIC otimiza para **predição** (minimizar erro de generalização esperado)
- BIC é **consistente**: seleciona o modelo verdadeiro quando $n \to \infty$
- BIC favorece modelos mais **simples** (parcimoniosos)

---

### (b) Graus de Liberdade Efetivos de Ridge

Pela decomposição em valores singulares $\mathbf{B} = \mathbf{U}\mathbf{S}\mathbf{V}^\top$ (autovalores de $\mathbf{B}^\top\mathbf{B}$: $\mu_j = s_j^2$):

$$\mathbf{H}_\lambda = \mathbf{U}\,\text{diag}\!\left(\frac{\mu_j}{\mu_j + \lambda}\right)\mathbf{U}^\top$$

$$df(\lambda) = \operatorname{tr}(\mathbf{H}_\lambda) = \sum_{j=1}^p \frac{\mu_j}{\mu_j + \lambda}$$

**Interpretação:** Cada termo $\frac{\mu_j}{\mu_j+\lambda} \in [0,1]$ é a fração do $j$-ésimo componente que Ridge preserva. Direções de alta variância ($\mu_j \gg \lambda$) contribuem $\approx 1$; direções de baixa variância ($\mu_j \ll \lambda$) contribuem $\approx 0$.

**Casos extremos:**
- $\lambda \to 0$: $df(0) = \sum 1 = p$ (OLS completo — $p$ graus de liberdade)
- $\lambda \to \infty$: $df(\infty) = \sum 0 = 0$ (modelo nulo — sem graus de liberdade)

Ridge com $\lambda > 0$ é **efetivamente um modelo com menos de $p$ graus de liberdade** — mesmo usando as mesmas $p$ funções de base.

---

### (c) Double Descent (Dupla Descida)

**Fenômeno clássico (U-shape):** Erro de teste cai com a complexidade até um mínimo, depois sobe (overfitting). Esperado pelo tradeoff bias-variância.

**Double descent:** Para $p \approx n$ ("interpolation threshold"), o erro de teste sobe acentuadamente (pior dos mundos: complexo mas não generaliza). Para $p \gg n$, o erro cai novamente:

```
Erro │╲         /╲
     │  ╲      /  ╲___________
     │    ╲__/
     └──────────────────────── complexidade
               ↑ p = n
```

**Por que acontece?** Para $p \gg n$, existem infinitas soluções que interpolam os dados. A descida de gradiente com inicialização em zero converge para a solução de **mínima norma** — que é implicitamente regularizada (análoga ao Ridge com $\lambda \to 0^+$). Esta solução tem boa generalização porque é a mais simples entre todas que ajustam o treino.

**Difere do tradeoff clássico** porque assume que entre múltiplas soluções de treino zero existe um viés de seleção implícito pelo otimizador — ausente nos modelos clássicos com solução única.

---
---
# EXERCÍCIO 8 — (10 pontos)
**Tema: L3 Avançado — Soft-Thresholding, Caminhos de Regularização e Descida de Coordenadas**

**(a)** Para **design ortogonal** ($\mathbf{B}^\top\mathbf{B} = I$), derive que a solução Lasso é o **operador de soft-thresholding**:

$$\hat{\theta}_j^\text{Lasso} = \operatorname{sign}(\hat{\theta}_j^\text{OLS})\cdot\max(|\hat{\theta}_j^\text{OLS}| - \lambda/2,\; 0)$$

Interprete o que esse operador faz a cada coeficiente. Compare com a solução Ridge para o mesmo caso. **(5)**

**(b)** Descreva o **caminho de regularização** (coeficientes em função de $\lambda$) para Ridge e Lasso. Qual é a diferença fundamental? Por que o caminho do Lasso tem "kinks" (quebras) enquanto o do Ridge é suave? **(3)**

**(c)** A **descida de coordenadas** (coordinate descent) é o algoritmo padrão para Lasso. Descreva a atualização de $\theta_j$ mantendo todos os outros fixos. **(2)**

## ✅ Resposta — Exercício 8

---

### (a) Soft-Thresholding para Design Ortogonal

Com $\mathbf{B}^\top\mathbf{B} = I$, o Lasso se decompõe em $p$ problemas independentes:

$$\min_{\theta} \|\mathbf{B}\theta - y\|^2 + \lambda\|\theta\|_1 = \min_{\theta} \sum_{j=1}^p \left[(\hat{\theta}_j^\text{OLS} - \theta_j)^2 + \lambda|\theta_j|\right]$$

onde $\hat{\theta}^\text{OLS} = \mathbf{B}^\top y$.

**Condição de otimalidade (subgradiente):**

$-2(\hat{\theta}_j - \theta_j) + \lambda\,\partial|\theta_j| \ni 0$

Três casos:
- $\theta_j > 0$: $\theta_j = \hat{\theta}_j - \lambda/2$ (válido se $\hat{\theta}_j > \lambda/2$)
- $\theta_j < 0$: $\theta_j = \hat{\theta}_j + \lambda/2$ (válido se $\hat{\theta}_j < -\lambda/2$)
- $|\hat{\theta}_j| \leq \lambda/2$: a cúspide em zero trava $\theta_j = 0$

$$\boxed{\hat{\theta}_j^\text{Lasso} = S_{\lambda/2}(\hat{\theta}_j^\text{OLS}) = \operatorname{sign}(\hat{\theta}_j^\text{OLS})\cdot\max(|\hat{\theta}_j^\text{OLS}| - \lambda/2,\; 0)}$$

**Interpretação:** $S_{\lambda/2}$ **zera** coeficientes pequenos ($|\hat\theta_j| \leq \lambda/2$) e **encolhe** os demais por $\lambda/2$ em direção a zero — nunca além.

**Comparação com Ridge** (design ortogonal): $\hat{\theta}_j^\text{Ridge} = \hat{\theta}_j^\text{OLS}/(1+\lambda)$ — encolhimento proporcional, **nunca zero exato**.

---

### (b) Caminhos de Regularização

**Ridge ($\lambda: 0 \to \infty$):**
- Todos os coeficientes decrescem **suavemente e monotonicamente** para 0
- Caminho contínuo e diferenciável em $\lambda$ (solução Ridge tem derivada analítica)
- Todos os coeficientes chegam a zero ao mesmo tempo ($\lambda \to \infty$)

**Lasso ($\lambda: 0 \to \infty$):**
- Coeficientes tornam-se zero **um a um** em valores críticos $\lambda_j^*$ distintos
- Caminho tem "kinks" nos $\lambda_j^*$ onde $\theta_j$ muda de sinal ou passa por zero
- **Por que kinks?** A norma $|\theta_j|$ não é diferenciável em $\theta_j = 0$. Quando $\theta_j$ chega a zero, a estrutura ativa do problema muda — o subgradiente salta — e a trajetória muda de direção.

---

### (c) Descida de Coordenadas para Lasso

**Algoritmo:** Atualiza uma coordenada de cada vez (mantendo as demais fixas) até convergência.

**Atualização de $\theta_j$:**

1. Calcula o **resíduo parcial** excluindo a $j$-ésima feature: $r_i^{(j)} = y_i - \sum_{k \neq j} \theta_k B_k(x_i)$

2. Aplica soft-threshold ao coeficiente OLS univariado:

$$\theta_j \leftarrow S_{\lambda/2}\!\left(\frac{\sum_i r_i^{(j)} B_j(x_i)}{\sum_i B_j(x_i)^2}\right)$$

**Vantagem:** Convergência garantida (função convexa). Coeficientes já em zero frequentemente permanecem em zero — custo quase nulo para features inativas.

---
---
# EXERCÍCIO 9 — (10 pontos)
**Tema: L4 Avançado — Inferência Bayesiana Completa e Bayes Empírico**

**(a)** O que distingue a **inferência Bayesiana completa** do estimador MAP? Escreva a expressão da distribuição posterior $p(\theta \mid D)$ e da **distribuição preditiva posterior** $p(y^* \mid x^*, D)$. Por que a distribuição preditiva é mais informativa do que uma estimativa pontual? **(4)**

**(b)** O que é **Bayes Empírico** (Empirical Bayes / Type-II ML)? Como ele difere do Bayes completo na escolha de $\lambda$? Escreva o critério de otimização. **(3)**

**(c)** O que é um **prior conjugado**? Dê um exemplo relevante para regressão linear com ruído Gaussiano. Qual é a vantagem computacional? Mostre a conexão com Ridge. **(3)**

## ✅ Resposta — Exercício 9

---

### (a) Inferência Bayesiana Completa vs MAP

**MAP:** $\hat{\theta}_\text{MAP} = \arg\max_\theta p(\theta \mid D)$ — retorna **um único ponto** (o modo da posterior). Ignora toda a incerteza ao redor.

**Bayesiano completo:** A **distribuição posterior** pelo Teorema de Bayes:

$$p(\theta \mid D) = \frac{p(D \mid \theta)\, p(\theta)}{p(D)} \propto p(D \mid \theta)\, p(\theta)$$

Não é um ponto — é uma distribuição completa sobre todos os $\theta$ plausíveis, codificando incerteza epistêmica.

**Distribuição preditiva posterior** para nova entrada $x^*$:

$$p(y^* \mid x^*, D) = \int p(y^* \mid x^*, \theta)\, p(\theta \mid D)\, d\theta$$

A predição é a **média ponderada** sobre todos os modelos, pesados pela posterior.

| Aspecto | MAP | Bayesiano completo |
|---------|-----|-----------------|
| Fornece | Uma predição $\hat{y}^*$ | Distribuição $p(y^*\mid x^*, D)$ |
| Incerteza | Nenhuma | Intervalo de credibilidade natural |
| Robustez | Não (só o modo) | Sim (integra sobre $\theta$) |

Para modelo linear Gaussiano com prior Gaussiano, a preditiva posterior tem forma analítica:

$$p(y^* \mid x^*, D) = \mathcal{N}(\mathbf{b}(x^*)^\top\hat{\mu},\;\sigma^2 + \mathbf{b}(x^*)^\top\hat{\Sigma}\,\mathbf{b}(x^*))$$

O segundo termo é a **incerteza epistêmica** — maior em regiões distantes dos dados.

---

### (b) Bayes Empírico

**Problema:** No Bayes padrão, $\lambda$ (ou $\tau^2$) é escolhido a priori — subjetivamente.

**Bayes Empírico:** Estima $\lambda$ a partir dos próprios dados maximizando a **evidência** (marginal likelihood), integrando $\theta$:

$$\hat{\lambda}_\text{EB} = \arg\max_\lambda p(D \mid \lambda) = \arg\max_\lambda \int p(D \mid \theta)\, p(\theta \mid \lambda)\, d\theta$$

| | Bayes Completo | Bayes Empírico |
|--|---------------|---------------|
| Como escolhe $\lambda$ | Hiperprior + integração | **Otimiza** a evidência |
| Filosofia | Totalmente Bayesiano | Híbrido (frequentista em $\lambda$) |
| Vantagem | Propagação completa da incerteza | Prático; evita especificar hiperpriors |

Para o modelo linear Gaussiano, a evidência tem forma analítica:

$$p(D \mid \lambda) = \mathcal{N}(y;\, \mathbf{0},\; \sigma^2 I + \tfrac{1}{\lambda}\mathbf{B}\mathbf{B}^\top)$$

---

### (c) Prior Conjugado e Conexão com Ridge

**Definição:** $p(\theta)$ é conjugado à likelihood $p(D \mid \theta)$ quando a posterior $p(\theta \mid D)$ pertence à **mesma família paramétrica** que o prior.

**Exemplo:** Para likelihood Gaussiana $y \mid \theta \sim \mathcal{N}(\mathbf{B}\theta, \sigma^2 I)$:
- Prior conjugado: $\theta \sim \mathcal{N}(\mu_0, \Sigma_0)$
- Posterior: $\theta \mid D \sim \mathcal{N}(\mu_n, \Sigma_n)$ com forma analítica:

$$\Sigma_n = (\Sigma_0^{-1} + \sigma^{-2}\mathbf{B}^\top\mathbf{B})^{-1}, \quad \mu_n = \Sigma_n(\Sigma_0^{-1}\mu_0 + \sigma^{-2}\mathbf{B}^\top y)$$

**Conexão com Ridge:** Para $\mu_0 = 0$ e $\Sigma_0 = (\sigma^2/\lambda)I$:

$$\mu_n = (\mathbf{B}^\top\mathbf{B} + \lambda I)^{-1}\mathbf{B}^\top y = \hat{\theta}_\text{Ridge}$$

O estimador Ridge é exatamente a média da posterior Bayesiana com prior Gaussiano isótropo.

**Vantagem computacional:** Sem prior conjugado, a posterior requer métodos numéricos como MCMC (Monte Carlo Markov Chain). Com prior conjugado, temos a **posterior analítica** — muito mais eficiente.

In [ ]:
# === Visualização — Bônus: Bootstrap e Cross-Validation ===

from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Bootstrap: distribuição de coeficientes
ax = axes[0]
np.random.seed(42)
N_b, B_rep = 60, 600
X_b, y_b = make_regression(n_samples=N_b, n_features=3, n_informative=2, noise=20, random_state=42)
X_b = StandardScaler().fit_transform(X_b)

theta_orig = Ridge(alpha=1.0).fit(X_b, y_b).coef_[0]
boot_coefs = [Ridge(alpha=1.0).fit(X_b[idx:=np.random.choice(N_b,N_b,replace=True)],
                                    y_b[idx]).coef_[0] for _ in range(B_rep)]
boot_coefs = np.array(boot_coefs)

ax.hist(boot_coefs, bins=35, color='#3498DB', alpha=0.75, density=True)
ci_lo, ci_hi = np.percentile(boot_coefs, [2.5, 97.5])
ax.axvline(theta_orig, color='black', lw=2.5, label=f'Estimativa original: {theta_orig:.2f}')
ax.axvline(ci_lo, color='orange', lw=2, ls='--')
ax.axvline(ci_hi, color='orange', lw=2, ls='--', label=f'IC 95% Percentile\n[{ci_lo:.2f}, {ci_hi:.2f}]')
ax.axvline(0, color='red', lw=1, ls=':', label='θ=0 (irrelevante?)')
ax.set_title(f'Bootstrap (B={B_rep}): IC do Coeficiente $\\theta_1$', fontweight='bold')
ax.set_xlabel('$\\theta_1$ (bootstrap)'); ax.set_ylabel('Densidade')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Padrão de splits dos 4 métodos de CV
ax = axes[1]
N_vis = 20
X_vis = np.arange(N_vis).reshape(-1,1)

mapa = np.ones((9, N_vis))  # 1=treino (verde), 0=val (vermelho)

# Hold-Out (1 linha)
idx_val = np.arange(15, 20)
mapa[0, idx_val] = 0

# K-Fold K=5 (5 linhas)
for i, (tr, vl) in enumerate(KFold(n_splits=5, shuffle=False).split(X_vis)):
    mapa[1+i, vl] = 0

# LOO (3 linhas amostradas)
for i, (_, vl) in enumerate(list(LeaveOneOut().split(X_vis))[:3]):
    mapa[6+i, vl] = 0

labels_rows = (['Hold-Out'] + [f'K-Fold F{i+1}' for i in range(5)] +
               ['LOO 1','LOO 2','LOO 3'])
ax.pcolor(mapa, cmap='RdYlGn', vmin=0, vmax=1, edgecolors='white', linewidth=0.8)
ax.set_yticks(np.arange(9)+0.5); ax.set_yticklabels(labels_rows, fontsize=8)
ax.set_xlabel('Índice da amostra')
ax.set_title('Padrões de Split CV\n(Verde=Treino, Vermelho=Validação)', fontweight='bold')
p1 = mpatches.Patch(color='green', label='Treino')
p2 = mpatches.Patch(color='red', label='Validação')
ax.legend(handles=[p1,p2], loc='lower right', fontsize=9)

# Variabilidade do erro estimado por método
ax = axes[2]
np.random.seed(3)
X_cv2, y_cv2 = make_regression(n_samples=100, n_features=8, n_informative=4, noise=20, random_state=3)
X_cv2 = StandardScaler().fit_transform(X_cv2)
modelo_cv = Ridge(alpha=1.0)

# Hold-Out 30 repetições
ho_scores = [-np.mean((y_cv2[idx_v:=np.random.choice(100,25,replace=False)] -
                        modelo_cv.fit(np.delete(X_cv2,idx_v,0), np.delete(y_cv2,idx_v)).predict(X_cv2[idx_v]))**2)
             for _ in range(30)]
kf5_scores  = cross_val_score(modelo_cv, X_cv2, y_cv2,
                               cv=KFold(5,shuffle=True,random_state=0), scoring='neg_mean_squared_error')
kf10_scores = cross_val_score(modelo_cv, X_cv2, y_cv2,
                               cv=KFold(10,shuffle=True,random_state=0), scoring='neg_mean_squared_error')
loo_scores  = cross_val_score(modelo_cv, X_cv2, y_cv2,
                               cv=LeaveOneOut(), scoring='neg_mean_squared_error')

resultados = {'Hold-Out\n(30 reps)': np.abs(ho_scores),
              '5-Fold CV': np.abs(kf5_scores),
              '10-Fold CV': np.abs(kf10_scores),
              f'LOO (N=100)': np.abs(loo_scores)}

nomes = list(resultados.keys())
ax.boxplot([v for v in resultados.values()], tick_labels=nomes, patch_artist=True,
           boxprops=dict(facecolor='#AED6F1'), medianprops=dict(color='navy', lw=2))
medias = [v.mean() for v in resultados.values()]
ax.scatter(range(1,5), medias, color='red', s=60, zorder=5, label='Média')
ax.set_title('Variabilidade do Erro Estimado\npor método de CV', fontweight='bold')
ax.set_ylabel('|MSE| por fold'); ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')

plt.suptitle('Exercício Bônus — Bootstrap e Cross-Validation',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
---
## Tabela Resumo — Guia de Estudo (100 pontos)

| Ex | Tema | Conceito-chave | Pts |
|----|------|----------------|-----|
| **1a-f** | L1 Básico | Ansatz, Equação Normal, Bases (Poly/RBF/Fourier), Singularidade, Vandermonde | 20 |
| **2a-f** | L2 Básico | Bias²+Var+σ², Bagging prova, Métodos de redução, Learning curves, 1-SE, Err. treino | 20 |
| **3a-f** | L3 Básico | Tikhonov, Ridge derivação, L1 vs L2 geom., Early Stopping, CV para λ, Elastic Net | 20 |
| **4a-e** | L4 Básico | MLE=MSE, MAP: Gaussiano→Ridge/Laplace→Lasso, MLE vs MAP, interp. λ, Convexidade | 20 |
| **5a-d** | Aplicado | p>n: Lasso/Ridge; Overfitting diagnose; Data snooping; Bootstrap vs CV | 10 |
| **6a-c** | L1+ | Hat matrix projeção; R² vs R²adj; Normalização de features antes de regularizar | 10 |
| **7a-c** | L2+ | AIC vs BIC; Graus de liberdade de Ridge df(λ)=Σμⱼ/(μⱼ+λ); Double descent | 10 |
| **8a-c** | L3+ | Soft-thresholding (design ortogonal); Caminho de regularização kinks; Coord. descent | 10 |
| **9a-c** | L4+ | Posterior preditivo; Bayes Empírico (evidência); Prior conjugado → Ridge | 10 |
| **Bônus** | B/CV | Bootstrap ICs (Normal/Percentil/BCa); CV 4 métodos; Bagging + correlação | 10 |
| | **Total** | | **90 + 10 bônus** |

---

## Dicas Finais

1. **Mostre sempre a derivação** — escrever só a fórmula final vale metade dos pontos.
2. **Conecte os conceitos** — ex: Prior Gaussiano → L2 porque $-\log\mathcal{N}(0,\tau^2) = \|\theta\|^2/2\tau^2$
3. **Use a geometria L1 vs L2** — bola vs diamante; tangência nos vértices → $\theta_j=0$ exato.
4. **Para CV** — K folds; retreinar com $\lambda^*$ em todo o treino; teste **uma única vez**.
5. **Normalizar antes de regularizar** — sempre, ou a penalidade L2 é injusta entre features.
6. **Bayesiano vs MAP** — MAP é um ponto; Bayesiano completo dá distribuição + incerteza epistêmica.
7. **Double descent** — para $p \gg n$, GD converge para solução de mínima norma → boa generalização.